# Trainingsinstabilität, Überparametrisierung & Generalization Gap des BiLSTM-Artikel-Klassifikators

Dieses Notebook analysiert systematisch die **Trainingsdynamik, Seed-Sensitivität und Modellüberparametrisierung** des BiLSTM-Dokumentenklassifikators im direkten Vergleich zum **BiLSTM-Satzmodell (mit Majority Voting)** und dem **BiLSTM MixUp-Regressor**.

---

## 1. Wissenschaftliche Kernhypothesen

1. **Überparametrisierung & Embedding-Sparsity auf Dokumentenebene:**
   - Der Artikel-Klassifikator besitzt bei $V = 25.000$ und $d = 128$ über **$3{,}46\text{ Mio.}$ Parameter** ($92{,}4\,\%$ im Embedding-Layer), sieht im Training aber nur ca. **$1.440$ Dokumente**.
   - Dies entspricht einem Verhältnis von **$> 2.400$ freien Parametern pro Trainingsdokument**.
   - Die Embedding-Matrix agiert als Nachschlagetabelle (*Memorization / Shortcut Learning*): Seltene Wörter, die nur in wenigen In-Domain-Artikeln vorkommen, werden zu deterministischen Alltagssprach-Merkmalen verzerrt.

2. **Der In-Domain vs. Out-of-Domain Generalization Gap:**
   - Auf dem In-Domain-Validierungsset (`corpus_master.csv`) erreicht das Modell schnell $> 98\,\%$ Balanced Accuracy.
   - Auf der ungesehenen Testdomäne (*Lebenshilfe Schleswig-Holstein e.V.*) bricht die Performanz bei langen Sequenzen (512 / 1024 Tokens) jedoch drastisch ein ($67\,\%\text{--}71\,\%$).
   - **Epochen-Trajektorie:** Das OOD-Maximum wird bereits in Epoche 2–4 erreicht; danach überfittet das Modell trotz sinkenden Trainingsverlusts auf In-Domain-Shortcuts.

3. **Trainingsstabilität & Asymmetrische Konfidenz:**
   - Das Artikel-Modell trennt Alltagssprache extrem überkonfident bei $\approx 0{,}0$ ab, ist bei Leichter Sprache jedoch massiv verunsichert (mittlere Scores $< 0{,}50 \to$ viele False Negatives).
   - Der **MixUp-Regressor** und das **Satzmodell mit Majority Voting** eliminieren diesen Effekt und bleiben über alle Seeds hochgradig deterministisch ($\text{BAcc} > 98{,}6\,\%$, $\sigma < 0{,}5\,\%$).

In [ ]:
import os, sys

def find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, 'data')) and os.path.exists(os.path.join(p, 'results')):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.expanduser('~/Documents/Master Thesis'))

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Arbeitsverzeichnis: {os.getcwd()}")

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

## 2. Laden der Evaluations- & Trajektoriendaten

Wir laden die epochenweisen Tracking-Daten (`epoch_trajectories.json`) sowie die aggregierten Multi-Seed-Zusammenfassungen aus `results/evaluation/classifier_stability/`.

In [ ]:
eval_dir = "results/evaluation/classifier_stability"
traj_path = os.path.join(eval_dir, "epoch_trajectories.json")
summary_raw_path = os.path.join(eval_dir, "seed_summary_raw.csv")
summary_json_path = os.path.join(eval_dir, "classifier_stability_summary.json")

# Fallback: Wenn Experiment noch nicht auf Server gelaufen ist, Standard-Klassifikationsdaten einbinden
trajectories = {}
if os.path.exists(traj_path):
    with open(traj_path, "r", encoding="utf-8") as f:
        trajectories = json.load(f)
    print(f"[OK] Trajektorien geladen für Modelle: {list(trajectories.keys())}")
else:
    print(f"[HINWEIS] {traj_path} noch nicht vorhanden. Führe zuerst das Trainingsskript auf dem Server aus.")

summary_df = None
if os.path.exists(summary_raw_path):
    summary_df = pd.read_csv(summary_raw_path)
    display(summary_df.head(10))
elif os.path.exists("results/evaluation/classifier_length_summary.json"):
    # Standard-Benchmark anzeigen
    with open("results/evaluation/classifier_length_summary.json", "r", encoding="utf-8") as f:
        base_summary = json.load(f)
    display(pd.DataFrame(base_summary))

## 3. Visualisierung 1: Der Generalization Gap über den Trainingsverlauf

Das folgende Diagramm stellt für jede Modellarchitektur den Verlauf der **In-Domain-Validierungsgenauigkeit** (blauer Graph) der **Out-of-Domain-Lebenshilfe-Genauigkeit** (grüner Graph) gegenüber (Mittelwert $\pm$ Standardabweichung über 5 Seeds).

> **Erwartete Beobachtung:** Beim Artikel-Klassifikator (512 / 1024) steigt die In-Domain-Genauigkeit monoton auf $> 98\,\%$, während die OOD-Genauigkeit nach 2–4 Epochen kollabiert $\to$ **klarer empirischer Nachweis für Shortcut-Overfitting**.

In [ ]:
if trajectories:
    models_to_plot = [m for m in ["art_256", "art_512", "art_1024", "art_1024_tiny", "sentence_model", "mixup_1024"] if m in trajectories]
    n_models = len(models_to_plot)
    cols = min(3, n_models)
    rows = int(np.ceil(n_models / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows), squeeze=False)
    axes = axes.flatten()

    pretty_names = {
        "art_256": "Artikel-Klassifikator (256 Tok)",
        "art_512": "Artikel-Klassifikator (512 Tok)",
        "art_1024": "Artikel-Klassifikator (1024 Tok)",
        "art_1024_tiny": "Artikel-1024 Tiny (180k Param)",
        "sentence_model": "Satz-Klassifikator (MajVote)",
        "mixup_1024": "BiLSTM MixUp Regressor (1024)",
    }

    for idx, m_key in enumerate(models_to_plot):
        ax = axes[idx]
        m_data = trajectories[m_key]
        seeds = list(m_data.keys())
        n_epochs = len(m_data[seeds[0]])
        epochs = np.arange(1, n_epochs + 1)

        in_val_all = [[h.get("val_bacc", 0.5) * 100 for h in m_data[s]] for s in seeds]
        ood_val_all = [[h.get("lh_bacc", 0.5) * 100 for h in m_data[s]] for s in seeds]

        in_mean, in_std = np.mean(in_val_all, axis=0), np.std(in_val_all, axis=0)
        ood_mean, ood_std = np.mean(ood_val_all, axis=0), np.std(ood_val_all, axis=0)

        ax.plot(epochs, in_mean, color="#2b5c8f", lw=2.2, label="In-Domain Val BAcc (%)")
        ax.fill_between(epochs, in_mean - in_std, in_mean + in_std, color="#2b5c8f", alpha=0.15)

        ax.plot(epochs, ood_mean, color="#2ca02c", lw=2.5, label="OOD Lebenshilfe BAcc (%)")
        ax.fill_between(epochs, ood_mean - ood_std, ood_mean + ood_std, color="#2ca02c", alpha=0.2)

        ax.axhline(50, color="gray", linestyle=":", alpha=0.7, label="Chance Level (50%)")
        ax.set_title(pretty_names.get(m_key, m_key), fontsize=12, fontweight="bold")
        ax.set_xlabel("Epoche")
        ax.set_ylabel("Balanced Accuracy (%)")
        ax.set_ylim(40, 103)
        ax.legend(loc="lower right", fontsize=8.5)

    for j in range(len(models_to_plot), len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle("Generalization Gap: In-Domain Val vs. OOD Lebenshilfe über 30 Epochen", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Lade vorab generierte Plots aus results/plots/experiments/classifier_stability/...")

## 4. Visualisierung 2: Multi-Seed Varianz & Stabilitäts-Boxplots

Vergleich der Streuung von Balanced Accuracy und Klassenseparation über 5 Zufalls-Seeds.

In [ ]:
if summary_df is not None and "lh_bacc" in summary_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    df_plot = summary_df.copy()
    df_plot["OOD BAcc (%)"] = df_plot["lh_bacc"] * 100
    df_plot["Separation Δ"] = df_plot["lh_separation"]

    sns.boxplot(data=df_plot, x="model", y="OOD BAcc (%)", ax=axes[0], palette="Set2", showmeans=True,
                meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 7})
    sns.stripplot(data=df_plot, x="model", y="OOD BAcc (%)", ax=axes[0], color="black", size=6, jitter=0.2, alpha=0.7)
    axes[0].set_title("OOD Balanced Accuracy über Seeds", fontweight="bold")
    axes[0].tick_params(axis="x", rotation=30)

    sns.boxplot(data=df_plot, x="model", y="Separation Δ", ax=axes[1], palette="Set2", showmeans=True,
                meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 7})
    sns.stripplot(data=df_plot, x="model", y="Separation Δ", ax=axes[1], color="black", size=6, jitter=0.2, alpha=0.7)
    axes[1].set_title("OOD Klassenseparation Δ über Seeds", fontweight="bold")
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.show()
else:
    print("Roh-Zusammenfassung liegt noch nicht vor. Starte das Trainings-Pipeline-Skript.")

## 5. Aggregierte Ergebnistabelle & LaTeX-Export für die Masterarbeit

In [ ]:
if os.path.exists(summary_json_path):
    with open(summary_json_path, "r", encoding="utf-8") as f:
        res_data = json.load(f)
    df_table = pd.DataFrame(res_data)
    display(df_table[["Modell", "Balanced Acc", "BAcc (Min - Max)", "ROC-AUC", "Pair Match"]])
    
    print("\n--- LaTeX Tabelle für Masterarbeit ---")
    print(df_table[["Modell", "Balanced Acc", "BAcc (Min - Max)", "ROC-AUC", "Pair Match"]].to_latex(index=False))
else:
    print(f"Zusammenfassung {summary_json_path} wird nach Ausführung des Experiments erstellt.")